# cjm-capability-qwen3-forced-aligner

> A Qwen3-based forced-alignment capability for the cjm-substrate runtime that provides word-level audio-text alignment using Qwen/Qwen3-ForcedAligner-0.6B with progress reporting.

## Install

```bash
pip install cjm_capability_qwen3_forced_aligner
```

## Project Structure

```
nbs/
└── capability.ipynb # Capability implementation for Qwen3 word-level forced alignment
```

Total: 1 notebook

## Module Dependencies

```mermaid
graph LR
    capability["capability<br/>Qwen3 Forced Aligner Capability"]

```

No cross-module dependencies detected.

## CLI Reference

No CLI commands found in this project.

## Module Overview

Detailed documentation for each module in the project:

### Qwen3 Forced Aligner Capability (`capability.ipynb`)
> Capability implementation for Qwen3 word-level forced alignment

#### Import

```python
from cjm_capability_qwen3_forced_aligner.capability import (
    Qwen3ForcedAlignerConfig,
    Qwen3ForcedAlignerCapability
)
```

#### Functions

```python
@patch
def is_available(self:Qwen3ForcedAlignerCapability) -> bool:  # True if qwen_asr is importable
    "Check if the Qwen3 forced aligner is available."
```

```python
@patch
def _apply_config(
    self:Qwen3ForcedAlignerCapability,
    config: Optional[Any] = None  # Configuration dataclass, dict, or None
) -> None
    """
    CR-4: apply config values only. Called by initialize (first-time) and the
    substrate's reconfigure delta path. Model release on a model_id/device/dtype/
    attn_implementation change is handled declaratively via RELOAD_TRIGGER ->
    _release_model (device/dtype are resolved lazily in _load_model).
    """
```

```python
@patch
def _load_model(self:Qwen3ForcedAlignerCapability):
    """Load model on first use. Heartbeat-wrapped HF Hub download + build."""
    if self._model is not None
    "Load model on first use. Heartbeat-wrapped HF Hub download + build."
```

```python
@patch
def _release_model(self:Qwen3ForcedAlignerCapability)
    """
    CR-4: release the model + free CUDA cache (cjm-substrate-torch-utils).
    RELOAD_TRIGGER target for model_id/device/dtype/attn_implementation; on_disable /
    cleanup delegate here. Idempotent (release_model no-ops when already None).
    """
```

```python
@patch
def prefetch(self:Qwen3ForcedAlignerCapability) -> None
    """
    CR-4 (SG-19): eagerly load the model so the first execute() doesn't pay
    the download/load cost. Idempotent via _load_model's None-guard.
    """
```

```python
@patch
def on_disable(self:Qwen3ForcedAlignerCapability) -> None
    """
    CR-2: release the GPU model when the operator disables the capability (the
    worker stays alive); lazy reload on the next execute after re-enable.
    """
```

```python
@patch
def cleanup(self:Qwen3ForcedAlignerCapability) -> None
    "Clean up resources."
```

#### Classes

```python
@dataclass
class Qwen3ForcedAlignerConfig(HFCacheConfig):
    """
    Configuration for the Qwen3 Forced Aligner capability.
    
    Composes HFCacheConfig (cache_dir / revision / local_files_only / trust_remote_code,
    each RELOAD_TRIGGER-tagged) so HF Hub download behaviour is operator-controllable.
    """
    
    model_id: str = field(...)
    device: str = field(...)
    dtype: str = field(...)
    attn_implementation: str = field(...)
    language: str = field(...)
```

```python
class Qwen3ForcedAlignerCapability:
    def __init__(self):
        """Initialize the capability with default state."""
        self.logger = logging.getLogger(f"{__name__}.{type(self).__name__}")
        self.config: Qwen3ForcedAlignerConfig = None
    """
    Qwen3 word-level forced-alignment tool capability (stage 8: pure compute).
    
    Native-surface model (PILLAR 1c): this tool is PURE COMPUTE — `align` reads
    MODEL-READY audio + the transcript text, runs Qwen3 inference, and builds the
    typed `ForcedAlignResult`. The cache-check + persistence bookends + the
    per-call `force` control live in the generic forced-alignment adapter
    (cjm-forced-alignment-adapter-interface); the result DTO lives in
    cjm-capability-primitives; identity derives from the installed distribution.
    No `get_plugin_metadata`, no `self.storage`.
    """
    
    def __init__(self):
            """Initialize the capability with default state."""
            self.logger = logging.getLogger(f"{__name__}.{type(self).__name__}")
            self.config: Qwen3ForcedAlignerConfig = None
        "Initialize the capability with default state."
    
    def name(self) -> str:  # Capability name identifier
            """Capability identity, derived from the installed distribution (PILLAR 1c)."""
            from importlib.metadata import metadata, packages_distributions
            dist = (packages_distributions().get(__package__) or [__package__.replace("_", "-")])[0]
            return metadata(dist)["Name"]
    
        @property
        def version(self) -> str:  # Capability version string
        "Capability identity, derived from the installed distribution (PILLAR 1c)."
    
    def version(self) -> str:  # Capability version string
            from cjm_capability_qwen3_forced_aligner import __version__
            return __version__
    
        def get_config_schema(self) -> Dict[str, Any]:  # JSON Schema for configuration
    
    def get_config_schema(self) -> Dict[str, Any]:  # JSON Schema for configuration
            """Return JSON Schema for UI generation."""
            return dataclass_to_jsonschema(Qwen3ForcedAlignerConfig)
    
        def get_current_config(self) -> Dict[str, Any]:  # Current configuration as dictionary
        "Return JSON Schema for UI generation."
    
    def get_current_config(self) -> Dict[str, Any]:  # Current configuration as dictionary
            """Return current configuration state."""
            return config_to_dict(self.config) if self.config else {}
    
        def initialize(
            self,
            config: Optional[Any] = None  # Configuration dataclass, dict, or None
        ) -> None
        "Return current configuration state."
    
    def initialize(
            self,
            config: Optional[Any] = None  # Configuration dataclass, dict, or None
        ) -> None
        "First-time setup. CR-4: config application is factored into _apply_config;
the substrate's reconfigure fires _release_model on a model_id/device/dtype/
attn_implementation change (RELOAD_TRIGGER) then re-applies config. No storage
init — the adapter owns the cache (stage 8)."
    
    def align(
            self,
            audio: Union[str, Path],  # Path to MODEL-READY audio (converted upstream)
            text: str,                # Transcript text to align against the audio
            **kwargs                  # Provenance pass-through (unused by FA compute)
        ) -> ForcedAlignResult:       # Word-level alignment result
        "Align transcript text to model-ready audio at word level — PURE COMPUTE.

Stage 8 / PILLAR 1c: the cache-check + persistence bookends + per-call
`force` moved to the generic forced-alignment adapter; this method loads
the model, runs Qwen3, and builds the typed result. The alignment language
comes from `self.config.language` (no per-call kwarg override — the tool
runs its effective config; the prior unhashed `language` override was
retired at migration)."
```
